# 01 · Reconnaissance automatique de la parole tunisienne (ASR)

**Projet commun GenAI / NLP · corpus TEDxTN**

**Objectifs**
1. Explorer et caractériser le corpus TEDxTN (durées, longueurs, répartition, code-switching).
2. Préparer des splits train / validation / test **sans fuite** (talks et locuteurs disjoints).
3. Mesurer une **baseline** zero-shot (Whisper), puis **adapter** deux familles de modèles :
   Whisper (encodeur-décodeur) et w2v-BERT-2.0 (encodeur auto-supervisé + CTC).
4. Évaluer avec WER / CER **bruts** (protocole du papier) et **normalisés**, avec intervalles de confiance.
5. Analyser les erreurs propres au tunisien : variations orthographiques, script, emprunts, prononciation.

**Améliorations par rapport au papier de référence** (Bougares et al., 2025), toutes implémentées ici :

| Limite identifiée | Ce que fait ce notebook |
|---|---|
| ~22 h d'entraînement seulement | augmentation audio (vitesse, bruit, gain) + SpecAugment |
| calcul limité (80 epochs sur HPC dans le papier) | LoRA, fp16, gradient checkpointing, early stopping |
| aucun modèle de langage | LM 4-gram KenLM pour le décodage CTC, paramètres réglés sur le dev |
| orthographe non standard, erreurs de script | WER normalisé + WER « script-aware » en plus du WER brut |
| dev/test petits (5 et 6 talks) | intervalles de confiance à 95 % par bootstrap |
| code-switching | WER par type de segment (TUN / MIXED / FOR) comme la Table 9 |
| volet GenAI | post-correction des transcriptions par un LLM (few-shot), évaluée |

## 1. Introduction

Le dialecte tunisien (derja) est une variété d'arabe très présente à l'oral mais peu dotée en ressources.
Trois difficultés le rendent dur pour la reconnaissance automatique :

- **Pas d'orthographe standard** : un même mot s'écrit de plusieurs façons (أنا / آنا / انا).
- **Code-switching massif** : 53,7 % des segments de TEDxTN contiennent au moins un mot étranger
  (surtout français), parfois **à l'intérieur d'un mot** (« الfumoir »).
- **Peu de données annotées** : TEDxTN apporte ~25 h transcrites et traduites.

Les modèles multilingues en zero-shot échouent (Whisper-large-v3 : 94 % de WER sur le test), alors qu'un
fine-tuning ramène le WER à 21-25 %. Ce notebook reproduit ce résultat, l'améliore et en analyse les erreurs.

> **Organisation en équipe.** Chaque section qui entraîne un modèle enregistre ses prédictions dans
> `results/preds/`. On peut donc exécuter les sections séparément (sur plusieurs comptes Colab) puis lancer
> la section 8 qui rassemble tout. Personne 1 : sections 2-5 ; personne 2 : sections 6-10.

## 2. Installation et configuration

Sur Colab : *Exécution > Modifier le type d'exécution > GPU*. Les données et modèles sont stockés sur
Google Drive (`MyDrive/tedxtn`) pour survivre aux déconnexions.

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/roudayn-kouka/tedxtn-tunisian-speech.git"  # à adapter au dépôt de l'équipe

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("TEDXTN_WORKDIR", "/content/drive/MyDrive/tedxtn")
    project = Path("/content/drive/MyDrive/tedxtn-tunisian-speech")      # dossier du projet copié sur Drive...
    if not project.exists():
        project = Path("/content/tedxtn-tunisian-speech")                # ...ou clone du dépôt
        if not project.exists():
            subprocess.run(["git", "clone", "-q", REPO_URL, str(project)], check=True)
    os.chdir(project)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, HTML, display

from common import data, metrics, text
from common.config import (DATA_DIR, LM_DIR, MODELS_DIR, PAPER_ASR_RESULTS, RESULTS_DIR, SEED, BASE_MODELS,
                           TrainConfig)

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    print("GPU :", torch.cuda.get_device_name(0) if HAS_GPU else "aucun (les sections d'entraînement seront ignorées)")
except ImportError:
    HAS_GPU = False

### Plan d'expériences
Les interrupteurs ci-dessous permettent de n'exécuter qu'une partie du notebook. Sur un GPU T4 (Colab gratuit),
Whisper-small en fine-tuning complet et w2v-BERT-2.0 tiennent en mémoire ; Whisper-large-v3 passe en LoRA.
Les temps d'entraînement se comptent en heures : prévoir plusieurs sessions (les checkpoints sont sur Drive).

In [ ]:
from common.config import REUSE_PREDS, SMOKE, env_flag, gpu_memory_gb, trained
VRAM = gpu_memory_gb()

# Chaque interrupteur peut être forcé sans modifier le notebook : TEDXTN_RUN_<NOM>=0/1 (ex. TEDXTN_RUN_CTC_FT=1).
DOWNLOAD = os.environ.get("TEDXTN_SKIP_DOWNLOAD") != "1"   # télécharger annotations + audio
RUN_BASELINES = env_flag("BASELINES", HAS_GPU)              # section 5
RUN_WHISPER_FT = env_flag("WHISPER_FT", HAS_GPU)            # section 6
RUN_CTC_FT = env_flag("CTC_FT", HAS_GPU and VRAM >= 12)     # section 7 : w2v-BERT-2.0 complet (~600 M paramètres)
                                                            # ne tient pas sous 12 Go -> Colab (notebook 01b)
RUN_MMS = env_flag("MMS", False)                            # option : MMS-1B (adaptateur)
RUN_AUG_ABLATION = env_flag("AUG_ABLATION", False)          # option : refaire Whisper sans augmentation
RUN_LLM_CORRECTION = env_flag("LLM_CORRECTION", HAS_GPU)    # section 10 (GenAI)

BASELINE_MODELS = ["whisper-small", "whisper-large-v3"]    # zero-shot (large-v3 en fp16 : ~3,2 Go, tient sur 6 Go)
WHISPER_FT_MODEL = "whisper-small"                          # "whisper-large-v3" + USE_LORA=True sur A100/L4
USE_LORA = WHISPER_FT_MODEL not in ("whisper-small",)
LLM_MODEL = BASE_MODELS["llm"] if VRAM >= 20 else BASE_MODELS["llm-small"]   # Qwen2.5-7B ou 1.5B selon la VRAM
GEN_BS = 16 if VRAM >= 12 else 4                            # batch d'inférence Whisper
N_EVAL = None   # None = split complet ; un entier pour un essai rapide
print(f"VRAM {VRAM:.1f} Go | SMOKE={SMOKE} | baselines={RUN_BASELINES} whisper_ft={RUN_WHISPER_FT} "
      f"ctc_ft={RUN_CTC_FT} llm={RUN_LLM_CORRECTION} ({LLM_MODEL}) | batch inférence {GEN_BS}")

## 3. Données
### 3.1 Chargement des annotations
Le dépôt Hugging Face contient, pour chaque split, deux CSV alignés segment par segment :
`<split>.csv` (transcription tunisienne) et `<split>.en.csv` (traduction anglaise). On les fusionne sur
(talk, début, fin) et on ajoute des colonnes dérivées (durée, nombre de mots, type de segment, CMI).

In [ ]:
df = data.load_corpus(download=DOWNLOAD)
print(f"{len(df):,} segments, {df.talk.nunique()} talks, {df.duration.sum() / 3600:.1f} h annotées")
df.head(3)

### 3.2 Vérification des splits et des fuites de données
Le papier place des **talks entiers** dans le dev et le test : aucun talk, donc aucun locuteur principal,
n'apparaît dans deux splits. On le vérifie, ainsi que l'absence de transcriptions dupliquées et de segments
qui se chevauchent. On compare aussi les effectifs à la Table 5 du papier.

In [ ]:
display(data.check_splits(df))
leak = data.check_leakage(df)
print(json.dumps(leak, ensure_ascii=False, indent=1))
assert leak["ok"], "Fuite de données entre splits : à corriger avant toute expérience."

### 3.3 Caractéristiques du corpus

In [ ]:
summary = df.groupby("split").agg(
    talks=("talk", "nunique"), evenements=("event", "nunique"), segments=("seg_id", "size"),
    heures=("duration", lambda x: round(x.sum() / 3600, 2)), duree_moy_s=("duration", "mean"),
    mots=("n_words", "sum"), mots_par_seg=("n_words", "mean"), cmi_moyen=("cmi", "mean"),
).reindex(["train", "dev", "test"]).round(2)
summary["cmi_papier"] = [20.66, 24.37, 33.09]
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for s, c in zip(["train", "dev", "test"], ["#c2410c", "#0f766e", "#7c3aed"]):
    d = df[df.split == s]
    axes[0].hist(d.duration, bins=60, range=(0, 30), alpha=.55, density=True, label=s, color=c)
    axes[1].hist(d.n_words, bins=60, range=(0, 80), alpha=.55, density=True, label=s, color=c)
    axes[2].hist((d.n_chars / d.duration.clip(lower=.1)), bins=60, range=(0, 30), alpha=.55, density=True, label=s, color=c)
axes[0].set_title("Durée des segments (s)"); axes[1].set_title("Mots par segment"); axes[2].set_title("Débit (caractères / s)")
axes[0].legend(); plt.tight_layout(); plt.savefig(RESULTS_DIR / "01_distributions.png", bbox_inches="tight"); plt.show()

**Lecture.** Les segments durent en moyenne ~5 s (papier : 5,2 s). La longue traîne au-delà de 30 s est
incompatible avec la fenêtre de Whisper : ces segments seront exclus **de l'entraînement seulement**.
Un débit anormalement élevé signale un alignement temporel douteux.

### 3.4 Particularités linguistiques : code-switching
On classe chaque mot selon son script (arabe = tunisien, latin = mot étranger, mixte = code-switching
intra-mot) et chaque segment en TUN / MIXED / FOR. On compare à la Table 3 du papier.

In [ ]:
cs = pd.DataFrame({s: text.code_switch_stats(df[df.split == s].transcription) for s in ["train", "dev", "test"]}).T
display(cs.round(2))
print(f"Segments avec au moins un mot étranger : {text.code_switch_stats(df.transcription)['pct_segments_with_foreign']:.1f} % (papier : 53,7 %)")
print(f"CMI du corpus : {text.corpus_cmi(df.transcription):.2f} % (papier : 21,50 %)")

In [ ]:
from collections import Counter
words = Counter(w for t in df[df.split == "train"].transcription for w in t.split())
lat = Counter({w: c for w, c in words.items() if text.token_script(w) == "lat"})
mix = Counter({w: c for w, c in words.items() if text.token_script(w) == "mixed"})
# mots déclencheurs : mot arabe juste avant un passage au latin (cf. Table 4 du papier)
trig = Counter()
for t in df[df.split == "train"].transcription:
    toks = t.split()
    for a, b in zip(toks, toks[1:]):
        if text.token_script(a) == "ar" and text.token_script(b) in ("lat", "mixed"):
            trig[a] += 1
display(pd.DataFrame({"mots étrangers fréquents": [f"{w} ({c})" for w, c in lat.most_common(15)],
                      "intra-mot fréquents": [f"{w} ({c})" for w, c in mix.most_common(15)] + [""] * max(0, 15 - len(mix.most_common(15))),
                      "déclencheurs de switch": [f"{w} ({c})" for w, c in trig.most_common(15)]}))

### 3.5 Variations orthographiques et vocabulaire
On regroupe les formes qui deviennent identiques après normalisation (alef, ya, diacritiques, allongements...).
Chaque groupe de plus d'une forme est une variation orthographique que le WER brut compte comme une erreur.

In [ ]:
groups = {}
for w, c in words.items():
    k = text.normalize_text(w)
    if k:
        groups.setdefault(k, Counter())[w] += c
variants = {k: v for k, v in groups.items() if len(v) > 1}
print(f"{len(words):,} formes distinctes dans le train → {len(groups):,} après normalisation "
      f"({100 * (1 - len(groups) / len(words)):.1f} % de réduction) ; {len(variants):,} groupes de variantes")
top = sorted(variants.items(), key=lambda kv: -sum(kv[1].values()))[:12]
display(pd.DataFrame([{"forme normalisée": k, "variantes (fréquence)": ", ".join(f"{w} ({c})" for w, c in v.most_common())} for k, v in top]))

train_vocab = set(words)
vocab_n = {w for t in df[df.split == "train"].transcription for w in text.normalize_text(t).split()}
for s in ["dev", "test"]:
    toks = [w for t in df[df.split == s].transcription for w in t.split()]
    oov = sum(w not in train_vocab for w in toks) / max(len(toks), 1)
    toks_n = [w for t in df[df.split == s].transcription for w in text.normalize_text(t).split()]
    oov_n = sum(w not in vocab_n for w in toks_n) / max(len(toks_n), 1)
    print(f"Taux de mots hors vocabulaire du train ({s}) : {100 * oov:.1f} % brut → {100 * oov_n:.1f} % normalisé")

## 4. Préparation des données
### 4.1 Récupération de l'audio
La licence CC BY-NC-ND interdit de redistribuer l'audio : le papier fournit l'URL YouTube de chaque talk.
`data.fetch_all_audio` cherche ces URL dans le dépôt, puis par titre sur YouTube en dernier recours.
**Contrôle qualité** : on vérifie que chaque audio dure au moins jusqu'à la dernière annotation ;
sinon, la vidéo trouvée n'est probablement pas la bonne et l'URL doit être corrigée à la main dans
`workdir/data/youtube_urls.csv`.

> Si YouTube bloque Colab (« confirm you're not a bot »), exporter les cookies du navigateur et définir
> `os.environ["YTDLP_COOKIES"] = "/content/drive/MyDrive/cookies.txt"`, ou télécharger les talks en local
> avec `python scripts/prepare_data.py` puis copier `workdir/audio_full/` sur Drive.

In [ ]:
if DOWNLOAD:
    report = data.fetch_all_audio(df)
    report.to_csv(DATA_DIR / "audio_report.csv", index=False)
    display(report[~report.coherent])
df = data.cut_segments(df)
print(df.groupby("split").size())

In [ ]:
ex = df[(df.split == "train") & (df.seg_type == "MIXED")].sample(1, random_state=1).iloc[0]
print(ex.transcription, "\n→", ex.translation)
display(Audio(ex.path))

### 4.2 Nettoyage pour l'entraînement
Règles appliquées au **train uniquement** (dev et test restent intacts pour une évaluation honnête) :
transcription vide, durée < 0,3 s, durée > 30 s (fenêtre Whisper), débit > 40 caractères/s.

In [ ]:
cfg = TrainConfig()
train_df, rejected = data.clean_for_training(df[df.split == "train"], cfg.min_audio_seconds, cfg.max_audio_seconds)
dev_df = df[df.split == "dev"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)
if SMOKE:
    # Essai rapide (TEDXTN_SMOKE=1) : uniquement des talks de TRAIN déjà découpés. Deux talks de train tenus à
    # part jouent le rôle de dev et de test : le dev et le test officiels ne sont pas utilisés.
    _tr = df[df.split == "train"]
    _talks = sorted(_tr.talk.unique())[:5]   # 5 premiers talks (ordre alphabétique) : même sélection d'un notebook à l'autre
    assert len(_talks) == 5, "smoke test : il faut au moins 5 talks de train découpés"
    dev_df = _tr[_tr.talk == _talks[3]].head(50).assign(split="dev").reset_index(drop=True)
    test_df = _tr[_tr.talk == _talks[4]].head(50).assign(split="test").reset_index(drop=True)
    train_df = train_df[train_df.talk.isin(_talks[:3])]
    train_df = train_df.sample(min(200, len(train_df)), random_state=SEED).reset_index(drop=True)
    print("SMOKE TEST :", {s: len(x) for s, x in [("train", train_df), ("pseudo-dev", dev_df), ("pseudo-test", test_df)]})
if N_EVAL:
    dev_df, test_df = dev_df.sample(min(N_EVAL, len(dev_df)), random_state=SEED), test_df.sample(min(N_EVAL, len(test_df)), random_state=SEED)
print(f"train : {len(train_df):,} gardés, {len(rejected)} rejetés")
display(rejected.raison.value_counts().rename("segments rejetés"))
train_df.to_parquet(DATA_DIR / "train_clean.parquet", index=False)

## 5. Méthodologie et baselines

### 5.1 Choix des modèles
| Modèle | Pourquoi | Paramètres |
|---|---|---|
| Whisper (small / large-v3) | ASR multilingue de référence, entraîné sur 680 k h dont de l'arabe ; architecture encodeur-décodeur qui apprend aussi l'orthographe | 244 M / 1,55 G |
| w2v-BERT-2.0 + CTC | meilleur modèle du papier (21,37 % WER) ; pré-entraîné sur 4,5 M h de 143 langues ; CTC = alignement monotone, peu d'hallucinations | 580 M |
| (option) MMS-1B | 1 100+ langues ; adaptateur léger (2 M paramètres) entraînable très vite | 1 G |

### 5.2 Protocole d'évaluation
- **WER/CER bruts** (`RAW_NORM`) : ponctuation retirée, casse latine ignorée. C'est le plus proche du protocole du papier.
- **WER/CER normalisés** (`EVAL_NORM`) : en plus, variantes **orthographiques** neutralisées (alef, ya, diacritiques,
  /g/, chiffres, allongements), sans retirer ni couper de mot. Mesure la qualité *lexicale* sans pénaliser l'absence
  d'orthographe standard.
- **WER sans hésitations** (`EVAL_NORM_NOHES`, colonne à part) : `EVAL_NORM` + hésitations (آ, آم, إم, أ, إ, أه, اه, euh...)
  retirées de la référence et de l'hypothèse, repérées sur la forme d'origine (« أم » est un mot, gardé). Peut être
  plus élevé que le WER normalisé : retirer des hésitations bien reconnues réduit surtout le dénominateur.
- La frontière arabe/latin (`الfumoir → ال fumoir`) est une convention d'écriture (notebook 03), pas une règle
  d'évaluation : elle est asymétrique (les références écrivent les emprunts en arabe, seules les hypothèses
  contiennent des mots hybrides, qu'elle coupe en deux erreurs).
- **WER script-aware** : une substitution « même mot, autre script » (circulation ↔ سيركيلاسيون) n'est pas comptée.
- **Intervalle de confiance à 95 %** par bootstrap sur les segments (1 000 tirages).
- Toutes les décisions (choix du checkpoint, paramètres du LM) sont prises sur le **dev** ; le test n'est utilisé qu'une fois, à la fin.

In [ ]:
from common import asr as A

def eval_wavs(frame):
    return [data.load_audio(p) for p in frame.path]

if RUN_BASELINES:
    test_wavs = eval_wavs(test_df)
    for name in BASELINE_MODELS:
        if data.have_preds("asr", f"{name}-zeroshot", ["test"]):
            print(name, ": prédictions existantes relues (TEDXTN_REUSE_PREDS=1)"); continue
        proc, model = A.load_whisper(BASE_MODELS[name], for_training=False)   # fp16 sur GPU
        try:
            hyps = A.whisper_generate(model, proc, test_wavs, task="transcribe", batch_size=GEN_BS)
            data.save_preds("asr", f"{name}-zeroshot", "test", test_df.seg_id, hyps, test_df.transcription)
            print(name, metrics.wer_cer(test_df.transcription.tolist(), hyps))
        finally:
            del model; torch.cuda.empty_cache()

**Attendu** (Table 6) : ~94 % de WER pour large-v3 et > 100 % pour small (insertions, hallucinations,
sortie en arabe standard ou en script latin). Le zero-shot n'est pas utilisable : il faut adapter le modèle.

## 6. Adaptation de Whisper
- Fine-tuning complet de Whisper-small (ou LoRA pour medium / large-v3 : seules les matrices d'attention
  reçoivent des adaptateurs de rang 32, ~1 % des paramètres).
- Langue forcée `arabic`, tâche `transcribe`.
- **Augmentation** à la volée (vitesse 0,9/1,1, bruit rose à 10-30 dB de SNR, gain) + **SpecAugment**.
- **Early stopping** sur la loss du dev (patience 3) au lieu de 80 epochs fixes.
- Le WER est aussi suivi sur 200 segments du dev pendant l'entraînement (courbe d'apprentissage).

In [ ]:
from common.augment import Augmenter

def run_whisper_ft(model_name, use_lora, augment=True, tag=""):
    c = TrainConfig(use_lora=use_lora, augment=augment)
    if "large" in model_name:
        c.batch_size, c.grad_accum = 8, 4
    out = MODELS_DIR / f"asr_{model_name}{'-lora' if use_lora else ''}{tag}"
    if trained(out):   # passe finale : le modèle existe, pas de réentraînement (TEDXTN_RETRAIN=1 pour forcer)
        print("modèle final trouvé, entraînement sauté :", out / "final")
        return out / "final"
    proc, model = A.load_whisper(BASE_MODELS[model_name], task="transcribe", use_lora=use_lora,
                                 load_in_8bit=use_lora and "large" in model_name, cfg=c)
    tr = A.SpeechDataset(train_df, augmenter=Augmenter(seed=SEED) if augment else None)
    dv = A.SpeechDataset(dev_df)
    sub = dev_df.sample(min(200, len(dev_df)), random_state=SEED)
    cb = A.GenerationWERCallback(lambda m, w: A.whisper_generate(m, proc, w, batch_size=GEN_BS),
                                 eval_wavs(sub), sub.transcription.tolist())
    args = A.training_args(out, c, seq2seq=True)
    trainer = A.make_trainer(model, args, tr, dv, A.WhisperCollator(proc), proc, c, [cb.as_callback()])
    last = max(out.glob("checkpoint-*"), default=None, key=lambda p: int(p.name.split("-")[1])) if out.exists() else None
    trainer.train(resume_from_checkpoint=str(last) if last else None)   # reprise après coupure Colab
    final = A.save_for_inference(trainer.model, proc, out / "final")
    pd.DataFrame(trainer.state.log_history).to_csv(RESULTS_DIR / f"train_log_{out.name}.csv", index=False)
    pd.DataFrame(cb.history).to_csv(RESULTS_DIR / f"dev_wer_curve_{out.name}.csv", index=False)
    return final

def predict_whisper(model_dir, frame, split, system, num_beams=1):
    if data.have_preds("asr", system, [split]):
        print(system, split, ": prédictions existantes relues"); return data.load_preds("asr", split)[system].hyp.tolist()
    proc, model = A.load_whisper(str(model_dir), for_training=False)
    model = model.half() if HAS_GPU else model
    hyps = A.whisper_generate(model, proc, eval_wavs(frame), batch_size=GEN_BS, num_beams=num_beams)
    data.save_preds("asr", system, split, frame.seg_id, hyps, frame.transcription)
    del model; torch.cuda.empty_cache()
    return hyps

if RUN_WHISPER_FT:
    wdir = run_whisper_ft(WHISPER_FT_MODEL, USE_LORA)
    sysname = f"{WHISPER_FT_MODEL}-ft{'-lora' if USE_LORA else ''}"
    for split, frame in [("dev", dev_df), ("test", test_df)]:
        predict_whisper(wdir, frame, split, sysname)
    predict_whisper(wdir, test_df, "test", sysname + "-beam5", num_beams=5)   # effet du beam search
    if RUN_AUG_ABLATION:
        wdir_noaug = run_whisper_ft(WHISPER_FT_MODEL, USE_LORA, augment=False, tag="-noaug")
        for split, frame in [("dev", dev_df), ("test", test_df)]:
            predict_whisper(wdir_noaug, frame, split, sysname + "-noaug")

In [ ]:
for f in sorted(RESULTS_DIR.glob("train_log_asr_*.csv")):
    log = pd.read_csv(f)
    fig, ax = plt.subplots(figsize=(8, 3))
    if "loss" in log:
        ax.plot(log.step, log["loss"], label="train loss", color="#c2410c", alpha=.7)
    if "eval_loss" in log:
        e = log.dropna(subset=["eval_loss"]); ax.plot(e.step, e.eval_loss, "o-", label="dev loss", color="#0f766e")
    curve = RESULTS_DIR / f.name.replace("train_log_", "dev_wer_curve_")
    if curve.exists() and curve.stat().st_size > 5:
        cw = pd.read_csv(curve); ax2 = ax.twinx(); ax2.plot(cw.step, cw.dev_WER_subset, "s--", color="#7c3aed", label="WER dev")
        ax2.set_ylabel("WER dev (%)")
    ax.set_title(f.stem.replace("train_log_", "")); ax.set_xlabel("step"); ax.legend(loc="upper right")
    plt.tight_layout(); plt.savefig(RESULTS_DIR / f"01_curve_{f.stem}.png", bbox_inches="tight"); plt.show()

## 7. Adaptation de w2v-BERT-2.0 (CTC) et modèle de langage KenLM
Le vocabulaire CTC est appris sur les caractères du train (arabe + latin + chiffres, `|` = espace).
Les diacritiques sont retirés des cibles (rares et incohérents) ; les autres distinctions orthographiques
sont gardées pour rester comparables aux références.

**Amélioration : modèle de langage.** Le papier décode sans LM. On estime un 4-gram KenLM sur les
transcriptions **du train** et on décode en beam search (pyctcdecode). Les poids α (LM) et β (bonus de
longueur) sont choisis par grille sur le dev.

In [ ]:
CTC_DIR = MODELS_DIR / "asr_w2v-bert-2.0"

if RUN_CTC_FT:
    A.build_ctc_vocab(train_df.transcription.tolist(), CTC_DIR)
    c = TrainConfig(batch_size=8, grad_accum=4, learning_rate=5e-5, max_epochs=15)
    proc, model = A.load_ctc(BASE_MODELS["w2v-bert-2.0"], CTC_DIR, c)
    tr = A.SpeechDataset(train_df, augmenter=Augmenter(seed=SEED))
    dv = A.SpeechDataset(dev_df)
    sub = dev_df.sample(min(200, len(dev_df)), random_state=SEED)
    cb = A.GenerationWERCallback(lambda m, w: A.ctc_transcribe(m, proc, w), eval_wavs(sub), sub.transcription.tolist())
    args = A.training_args(CTC_DIR, c, seq2seq=False)
    trainer = A.make_trainer(model, args, tr, dv, A.CTCCollator(proc), proc, c, [cb.as_callback()])
    last = max(CTC_DIR.glob("checkpoint-*"), default=None, key=lambda p: int(p.name.split("-")[1]))
    trainer.train(resume_from_checkpoint=str(last) if last else None)
    A.save_for_inference(trainer.model, proc, CTC_DIR / "final")
    pd.DataFrame(trainer.state.log_history).to_csv(RESULTS_DIR / "train_log_asr_w2v-bert-2.0.csv", index=False)
    pd.DataFrame(cb.history).to_csv(RESULTS_DIR / "dev_wer_curve_asr_w2v-bert-2.0.csv", index=False)

In [ ]:
# Corpus du LM : transcriptions du TRAIN uniquement, avec la même normalisation que les cibles CTC
lm_corpus = LM_DIR / "train_ctc.txt"
lm_corpus.write_text("\n".join(text.normalize_text(t, A.CTC_NORM) for t in train_df.transcription), encoding="utf-8")
if RUN_CTC_FT and not (LM_DIR / "tedxtn_4gram.bin").exists():
    subprocess.run(["bash", "scripts/build_kenlm.sh", str(lm_corpus), str(LM_DIR), "4"], check=True)

In [ ]:
if RUN_CTC_FT:
    proc, model = A.load_ctc(str(CTC_DIR / "final"), None, from_checkpoint=True)
    dev_w, test_w = eval_wavs(dev_df), eval_wavs(test_df)
    # 1) décodage glouton (= protocole du papier)
    for split, frame, w in [("dev", dev_df, dev_w), ("test", test_df, test_w)]:
        hyps = A.ctc_transcribe(model, proc, w)
        data.save_preds("asr", "w2v-bert-2.0-ft", split, frame.seg_id, hyps, frame.transcription)
    # 2) grille α/β sur un sous-ensemble du dev
    sub_idx = np.random.default_rng(SEED).choice(len(dev_df), min(300, len(dev_df)), replace=False)
    grid = []
    for alpha in [0.3, 0.5, 0.8, 1.0]:
        for beta in [0.0, 1.0, 2.0]:
            dec = A.build_kenlm_decoder(proc, LM_DIR / "tedxtn_4gram.bin", alpha, beta)
            h = A.ctc_transcribe(model, proc, [dev_w[i] for i in sub_idx], decoder=dec)
            grid.append({"alpha": alpha, "beta": beta,
                         "WER_dev": metrics.wer_cer(dev_df.transcription.iloc[sub_idx].tolist(), h)["WER"]})
    grid = pd.DataFrame(grid).sort_values("WER_dev"); display(grid.head())
    grid.to_csv(RESULTS_DIR / "01_kenlm_grid.csv", index=False)
    best = grid.iloc[0]
    dec = A.build_kenlm_decoder(proc, LM_DIR / "tedxtn_4gram.bin", best.alpha, best.beta)
    for split, frame, w in [("dev", dev_df, dev_w), ("test", test_df, test_w)]:
        hyps = A.ctc_transcribe(model, proc, w, decoder=dec)
        data.save_preds("asr", "w2v-bert-2.0-ft+kenlm", split, frame.seg_id, hyps, frame.transcription)
    del model; torch.cuda.empty_cache()

### 7.1 (Option) MMS-1B avec adaptateur
Seul un adaptateur de langue (~2,5 M paramètres) est entraîné : très rapide, utile pour comparer une
adaptation légère à un fine-tuning complet.

In [ ]:
if RUN_MMS:
    mdir = MODELS_DIR / "asr_mms-1b"
    A.build_ctc_vocab(train_df.transcription.tolist(), mdir)
    c = TrainConfig(batch_size=8, grad_accum=2, learning_rate=1e-3, max_epochs=6, gradient_checkpointing=False)
    proc, model = A.load_ctc(BASE_MODELS["mms-1b-all"], mdir, c)
    trainer = A.make_trainer(model, A.training_args(mdir, c, seq2seq=False), A.SpeechDataset(train_df, augmenter=Augmenter()),
                             A.SpeechDataset(dev_df), A.CTCCollator(proc), proc, c)
    trainer.train()
    A.save_for_inference(trainer.model, proc, mdir / "final")
    for split, frame in [("dev", dev_df), ("test", test_df)]:
        data.save_preds("asr", "mms-1b-adapter", split, frame.seg_id, A.ctc_transcribe(trainer.model, proc, eval_wavs(frame)), frame.transcription)

## 8. Résultats
Cette section rassemble **toutes** les prédictions disponibles dans `results/preds/` (y compris celles
produites par d'autres membres de l'équipe) et calcule les métriques sur le test.

In [ ]:
def evaluate_split(split):
    rows = []
    for name, p in data.load_preds("asr", split).items():
        refs, hyps = p.ref.tolist(), p.hyp.tolist()
        raw = metrics.wer_cer(refs, hyps, text.RAW_NORM)
        norm = metrics.wer_cer(refs, hyps, text.EVAL_NORM)          # orthographe seule
        nohes = metrics.wer_cer(refs, hyps, text.EVAL_NORM_NOHES)   # + hésitations retirées (colonne à part)
        sa = metrics.script_aware_wer(refs, hyps)
        m, lo, hi = metrics.bootstrap_ci(metrics.wer_fn(text.RAW_NORM), hyps, refs, n_samples=500)
        rows.append({"système": name, "WER": raw["WER"], "IC95 WER": f"[{lo:.1f} ; {hi:.1f}]", "CER": raw["CER"],
                     "WER normalisé": norm["WER"], "CER normalisé": norm["CER"],
                     "WER sans hésitations": nohes["WER"], "WER script-aware": sa["WER_script_aware"],
                     "sub%": raw["sub%"], "del%": raw["del%"], "ins%": raw["ins%"], "segments": raw["n"]})
    return pd.DataFrame(rows).sort_values("WER").round(2) if rows else pd.DataFrame()

asr_dev = evaluate_split("dev")
asr_test = evaluate_split("test")
if len(asr_test):
    asr_test.to_csv(RESULTS_DIR / "asr_results.csv", index=False)
    display(asr_test)
    paper = pd.DataFrame([{"système (papier)": k, **v} for k, v in PAPER_ASR_RESULTS.items()])
    display(paper)
else:
    print("Aucune prédiction trouvée : exécuter les sections 5 à 7 (ou récupérer results/preds/ de l'équipe).")

In [ ]:
if len(asr_test):
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(asr_test) + 1))
    y = np.arange(len(asr_test))
    ax.barh(y - .2, asr_test["WER"], .4, label="WER brut", color="#c2410c")
    ax.barh(y + .2, asr_test["WER normalisé"], .4, label="WER normalisé", color="#0f766e")
    ax.set_yticks(y, asr_test["système"]); ax.invert_yaxis(); ax.set_xlabel("WER test (%)")
    ax.axvline(21.37, ls="--", color="#64748b"); ax.text(21.6, -0.6, "meilleur papier (21,37)", fontsize=8, color="#64748b")
    ax.legend(); plt.tight_layout(); plt.savefig(RESULTS_DIR / "01_wer_comparison.png", bbox_inches="tight"); plt.show()

**Comment lire le tableau.**
- L'écart *zero-shot → fine-tuné* mesure l'apport de l'adaptation au domaine et au dialecte.
- L'écart *WER brut → WER normalisé* mesure la part d'« erreurs » qui ne sont que des variantes d'orthographe.
- La colonne *WER sans hésitations* ne compte plus les hésitations, ni comme erreurs ni comme mots de référence.
- L'écart *normalisé → script-aware* mesure les erreurs dues au choix du script pour les emprunts.
- Deux systèmes dont les intervalles de confiance se recouvrent largement ne sont pas significativement
  différents sur ce test de 6 talks.

## 9. Analyse des erreurs
On analyse le meilleur système (WER normalisé le plus bas sur le **dev**).

In [ ]:
# Le système analysé (et exporté pour l'interface) est choisi sur le DEV uniquement, jamais sur le test.
# Seuls les systèmes fine-tunés, évalués sur le dev, sont candidats.
BEST = asr_dev.sort_values("WER normalisé").iloc[0]["système"] if len(asr_dev) else None
print("Système analysé (meilleur WER normalisé sur le dev) :", BEST if BEST else "aucun : pas de prédiction dev")
if BEST:
    P = data.load_preds("asr", "test")[BEST].merge(test_df[["seg_id", "duration", "cmi", "seg_type", "talk"]], on="seg_id")
    refs, hyps = P.ref.tolist(), P.hyp.tolist()

### 9.1 Par type de segment (reproduction de la Table 9)

In [ ]:
if BEST:
    t9 = metrics.wer_by_segment_type(refs, hyps, text.RAW_NORM).merge(
        metrics.wer_by_segment_type(refs, hyps, text.EVAL_NORM)[["type", "WER"]].rename(columns={"WER": "WER normalisé"}), on="type").merge(
        metrics.wer_by_segment_type(refs, hyps, text.EVAL_NORM_NOHES)[["type", "WER"]].rename(columns={"WER": "WER sans hésitations"}), on="type")
    t9["WER papier (w2v-BERT)"] = t9.type.map({"TUN": 24.16, "MIXED": 21.31, "FOR": 13.93, "ALL": 21.37})
    display(t9.round(2))

### 9.2 Nature des erreurs
Chaque substitution est classée automatiquement :
*variante orthographique* (identique après normalisation), *même mot, autre script*, *mot emprunté / code-switching*,
*proche phonétiquement* (prononciation, mots proches) ou *autre (lexical)*.

In [ ]:
if BEST:
    errs = metrics.error_table(refs, hyps, text.RAW_NORM)
    cat = errs.categorie.value_counts()
    display((100 * cat / cat.sum()).round(1).rename("% des erreurs"))
    fig, ax = plt.subplots(figsize=(8, 3.2)); cat.sort_values().plot.barh(ax=ax, color="#c2410c")
    ax.set_title(f"Catégories d'erreurs · {BEST}"); plt.tight_layout(); plt.savefig(RESULTS_DIR / "01_error_categories.png", bbox_inches="tight"); plt.show()
    subs = errs[errs.op == "S"]
    display(subs.groupby(["ref", "hyp", "categorie"]).size().sort_values(ascending=False).head(20).rename("fréquence").reset_index())

In [ ]:
if BEST:
    for c in ["même mot, autre script", "variante orthographique", "mot emprunté / code-switching", "proche phonétiquement (prononciation)"]:
        ex_ = subs[subs.categorie == c].head(5)
        print(f"\n— {c} —")
        for r in ex_.itertuples():
            print(f"   réf: {r.ref:<20} hyp: {r.hyp}")

### 9.3 Erreurs selon la durée, le taux de code-switching et le talk

In [ ]:
if BEST:
    P["wer_seg"] = [metrics.segment_wer(r, h) for r, h in zip(refs, hyps)]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
    P.groupby(pd.cut(P.duration, [0, 2, 4, 7, 12, 60])).wer_seg.mean().plot.bar(ax=axes[0], color="#c2410c")
    axes[0].set_title("WER moyen par durée (s)")
    P.groupby(pd.cut(P.cmi, [-1, 0, 15, 30, 50, 100])).wer_seg.mean().plot.bar(ax=axes[1], color="#0f766e")
    axes[1].set_title("WER moyen par CMI (%)")
    P.groupby("talk").wer_seg.mean().sort_values().plot.barh(ax=axes[2], color="#7c3aed")
    axes[2].set_title("WER moyen par talk (test)"); axes[2].tick_params(axis="y", labelsize=6)
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "01_error_breakdown.png", bbox_inches="tight"); plt.show()

### 9.5 Talks difficiles repérés au contrôle d'alignement

Le contrôle d'alignement (`scripts/check_alignment.py`, Whisper zero-shot) a signalé 21 talks au CER anormalement
élevé sans décalage démontré, dont *Pink Banana* (test), *The harmony in chaos* (dev) et 5 talks de TEDx Pasha Street
(train). On vérifie si le modèle fine-tuné y fait aussi beaucoup plus d'erreurs que la moyenne, et quels facteurs
l'expliquent : **bruit** (rapport signal/bruit estimé sur l'énergie par trames de 25 ms : 90e centile / 10e centile),
**débit** (caractères par seconde), **code-switching** (CMI, part de mots en script latin) et **accent** (région de
l'événement, notebook 04).

Pour les talks du train, le modèle les a vus pendant l'entraînement : l'erreur absolue est optimiste, on ne lit que
l'écart relatif avec des talks témoins du train (un talk qui reste très au-dessus des autres même après
entraînement signale un problème d'audio ou d'annotation).

In [ ]:
HARD_EXTRA = {"Pink_Banana_Good_News_from_Tunisia_Raya_Ben_Guiza_at_TEDxCarthageWomen", "The_harmony_in_chaos_Belkis_Anane_TEDxDjerba"}
_al = RESULTS_DIR / "alignment_summary.csv"
hard = set(pd.read_csv(_al).query("decision != 'OK'").talk) | HARD_EXTRA if _al.exists() else HARD_EXTRA
reg_f = RESULTS_DIR / "04_region_attribution.csv"
region = dict(pd.read_csv(reg_f)[["talk", "region"]].values) if reg_f.exists() else {}

def snr_db(path, win=0.025):
    import soundfile as sf
    w, sr = sf.read(path, dtype="float32"); n = int(win * sr)
    if len(w) < 4 * n:
        return np.nan
    e = (w[: len(w) // n * n].reshape(-1, n) ** 2).mean(1) + 1e-10
    return float(10 * np.log10(np.percentile(e, 90) / np.percentile(e, 10)))

def latin_share(t):
    tags = [s for _, s in text.tag_tokens(t)]
    return 100 * np.mean([s in ("lat", "mixed") for s in tags]) if tags else 0.0

def talk_table(frame, hyps):
    f = frame.assign(hyp=hyps)
    rows = []
    for t, g in f.groupby("talk"):
        m = metrics.wer_cer(g.transcription.tolist(), g.hyp.tolist())
        rows.append({"talk": t, "split": g.split.iloc[0], "event": g.event.iloc[0], "région": region.get(t),
                     "difficile (alignement)": t in hard, "segments": len(g), "WER": m["WER"], "CER": m["CER"],
                     "débit (car/s)": (g.n_chars / g.duration).mean(), "CMI": g.cmi.mean(),
                     "mots latins (%)": np.mean([latin_share(x) for x in g.transcription]),
                     "SNR estimé (dB)": np.nanmean([snr_db(p) for p in g.path])})
    return pd.DataFrame(rows)

hard_tab = None
if BEST:
    # dev et test : prédictions du système retenu sur le dev
    parts = []
    for split, frame in [("dev", dev_df), ("test", test_df)]:
        Pp = data.load_preds("asr", split).get(BEST)
        if Pp is not None:
            fr = frame.set_index("seg_id").loc[Pp.seg_id].reset_index()
            parts.append(talk_table(fr, Pp.hyp.tolist()))
    # train : talks signalés + 20 talks témoins, 40 segments chacun, transcrits par le système retenu
    tr_hard = sorted(t for t in hard if t in set(train_df.talk))
    rng_t = np.random.default_rng(SEED)
    _pool = sorted(set(train_df.talk) - hard)
    controls = sorted(rng_t.choice(_pool, min(20, len(_pool)), replace=False)) if _pool else []
    tr_s = train_df[train_df.talk.isin(tr_hard + controls)].groupby("talk", group_keys=False).apply(
        lambda x: x.sample(min(40, len(x)), random_state=SEED)).reset_index(drop=True)
    best_dir = {"w2v": CTC_DIR / "final"}.get(BEST[:3], MODELS_DIR / f"asr_{WHISPER_FT_MODEL}{'-lora' if USE_LORA else ''}" / "final")
    if "w2v" in BEST:
        proc, model = A.load_ctc(str(best_dir), None, from_checkpoint=True); h_tr = A.ctc_transcribe(model, proc, eval_wavs(tr_s))
    else:
        proc, model = A.load_whisper(str(best_dir), for_training=False); h_tr = A.whisper_generate(model, proc, eval_wavs(tr_s), batch_size=GEN_BS)
    del model; torch.cuda.empty_cache()
    parts.append(talk_table(tr_s, h_tr))
    hard_tab = pd.concat(parts, ignore_index=True)
    hard_tab.round(2).to_csv(RESULTS_DIR / "01_hard_talks.csv", index=False)
    # écart à la moyenne de son split (le train est vu à l'entraînement : seul l'écart relatif a un sens)
    hard_tab["WER - moyenne du split"] = hard_tab.WER - hard_tab.groupby("split").WER.transform("mean")
    show = ["split", "talk", "event", "région", "segments", "WER", "WER - moyenne du split", "CER", "débit (car/s)",
            "CMI", "mots latins (%)", "SNR estimé (dB)"]
    print("Talks signalés à l'alignement :")
    display(hard_tab[hard_tab["difficile (alignement)"]][show].sort_values(["split", "WER"], ascending=[True, False]).round(1))
    print("Moyennes par split : talks signalés vs autres")
    display(hard_tab.groupby(["split", "difficile (alignement)"])[["WER", "CER", "débit (car/s)", "CMI", "mots latins (%)", "SNR estimé (dB)"]].mean().round(1))
    print("Corrélation de Spearman avec le WER du talk (tous splits, n =", len(hard_tab), "talks) :")
    display(hard_tab[["WER", "débit (car/s)", "CMI", "mots latins (%)", "SNR estimé (dB)"]].corr("spearman")["WER"].drop("WER").round(2))
    pasha = hard_tab[hard_tab.event == "TEDxPashaStreet"]
    if len(pasha):
        print(f"TEDx Pasha Street ({len(pasha)} talks) : WER moyen {pasha.WER.mean():.1f} contre "
              f"{hard_tab[(hard_tab.split == 'train') & (hard_tab.event != 'TEDxPashaStreet')].WER.mean():.1f} pour les autres talks du train échantillonnés ; "
              f"SNR {pasha['SNR estimé (dB)'].mean():.1f} dB contre {hard_tab[hard_tab.event != 'TEDxPashaStreet']['SNR estimé (dB)'].mean():.1f} dB.")

### 9.4 Exemples réels (meilleurs et pires segments)

In [ ]:
from common.pipeline import diff_html
if BEST:
    show = pd.concat([P[P.ref.str.split().str.len() >= 5].nsmallest(3, "wer_seg"), P.nlargest(5, "wer_seg")])
    rows = "".join(f"<tr><td>{r.wer_seg:.0f} %</td><td>{r.seg_type}</td><td dir='rtl'>{r.ref}</td><td>{diff_html(r.ref, r.hyp)}</td></tr>"
                   for r in show.itertuples())
    display(HTML("<style>del{background:#fee2e2;color:#b91c1c}ins{background:#dcfce7;color:#166534;text-decoration:none}</style>"
                 f"<table><tr><th>WER</th><th>type</th><th>référence</th><th>hypothèse (diff)</th></tr>{rows}</table>"))

## 10. Volet GenAI : post-correction par LLM
Un LLM instruct (Qwen2.5-7B en 4 bits) reçoit la sortie ASR et doit corriger les erreurs évidentes sans
reformuler. Les exemples few-shot sont des paires (sortie ASR, référence) **du train**. Un garde-fou rejette
toute sortie trop éloignée de l'entrée (similarité < 0,6). On mesure le WER avant/après sur le test.

In [ ]:
if RUN_LLM_CORRECTION and BEST:
    from common import llm as L
    # few-shot : le meilleur modèle Whisper/CTC transcrit 40 segments du train
    best_dir = {"w2v": CTC_DIR / "final"}.get(BEST[:3], MODELS_DIR / f"asr_{WHISPER_FT_MODEL}{'-lora' if USE_LORA else ''}" / "final")
    shots_df = train_df.sample(min(40, len(train_df)), random_state=SEED)
    if "w2v" in BEST:
        proc, model = A.load_ctc(str(best_dir), None, from_checkpoint=True); shots_h = A.ctc_transcribe(model, proc, eval_wavs(shots_df))
    else:
        proc, model = A.load_whisper(str(best_dir), for_training=False)
        model = model.half() if HAS_GPU else model
        shots_h = A.whisper_generate(model, proc, eval_wavs(shots_df), batch_size=GEN_BS)
    del model; torch.cuda.empty_cache()
    pairs = [(h, r) for h, r in zip(shots_h, shots_df.transcription) if h != r and 4 <= len(r.split()) <= 25][:6]
    data.save_json(pairs, DATA_DIR / "llm_fewshot.json")   # réutilisé par l'interface
    tok, llm_model = L.load_llm(LLM_MODEL)
    llm_eval = {}
    for split in ["dev", "test"]:
        Pp = data.load_preds("asr", split)[BEST]
        r_, h_ = Pp.ref.tolist(), Pp.hyp.tolist()
        if data.have_preds("asr", f"{BEST}+llm", [split]):
            Pc = data.load_preds("asr", split)[f"{BEST}+llm"]; corrected, accepted = Pc.hyp.tolist(), Pc.accepted.tolist()
        else:
            corrected, accepted = L.post_correct(tok, llm_model, h_, pairs)
            data.save_preds("asr", f"{BEST}+llm", split, Pp.seg_id, corrected, r_, accepted=accepted)
        before, after = metrics.wer_cer(r_, h_), metrics.wer_cer(r_, corrected)
        llm_eval[split] = (before, after, metrics.paired_bootstrap(metrics.wer_fn(), h_, corrected, r_, n_samples=1000))
        print(f"[{split}] corrections acceptées : {np.mean(accepted) * 100:.1f} % des segments | "
              f"WER {before['WER']:.2f} → {after['WER']:.2f} | CER {before['CER']:.2f} → {after['CER']:.2f}")
        print(f"[{split}] différence de WER (corrigé - brut) :", llm_eval[split][2])
    # Décision prise sur le dev : la post-correction n'est retenue que si elle baisse le WER du dev de façon
    # significative (IC95 bootstrap de la différence entièrement sous 0). Le test est seulement rapporté.
    LLM_HELPS = llm_eval["dev"][2]["IC95"][1] < 0
    print("Post-correction retenue (décision sur le dev) :", LLM_HELPS)
    del llm_model; torch.cuda.empty_cache()

## 11. Export du meilleur modèle (pour les notebooks 02, 03 et l'interface)
Le meilleur système sur le dev est copié dans `workdir/models/asr_best` ; ses prédictions de test sont
celles utilisées par le pipeline en deux étapes du notebook 02.

In [ ]:
import shutil
if BEST:
    src = CTC_DIR / "final" if BEST.startswith("w2v") else MODELS_DIR / f"asr_{WHISPER_FT_MODEL}{'-lora' if USE_LORA else ''}" / "final"
    if src.exists():
        shutil.copytree(src, MODELS_DIR / "asr_best", dirs_exist_ok=True)
        (MODELS_DIR / "asr_best" / "SOURCE.txt").write_text(f"{BEST}\n")
        print("Modèle de l'interface :", MODELS_DIR / "asr_best", "←", BEST)

## 12. Discussion critique et limites

**Résultats obtenus** (à lire dans le tableau de la section 8, générés par ce notebook) et **améliorations
envisagées** sont séparés ci-dessous.

**Limites des données**
- ~22 h d'entraînement, 117 locuteurs, 11 villes, parole préparée (TEDx) : la généralisation à la parole
  spontanée, téléphonique ou bruitée n'est pas garantie.
- Déséquilibre de genre (77 H / 40 F dans le train) : un WER par genre serait utile mais le genre n'est pas
  annoté par segment dans les CSV publics.
- Dev et test minuscules (10 et 7 locuteurs) : les intervalles de confiance sont larges, un point de WER
  n'est souvent pas significatif.

**Limites de l'annotation et de la métrique**
- Pas d'orthographe standard : le WER brut surestime les erreurs. Le WER normalisé et le WER script-aware
  corrigent en partie ce biais, mais nos règles de normalisation sont elles-mêmes des conventions discutables
  (cf. notebook 03).
- Les emprunts peuvent être transcrits en arabe ou en latin selon l'annotateur : la référence elle-même est
  incohérente par endroits.

**Limites de calcul**
- Fine-tuning limité à quelques epochs (early stopping) au lieu de 80 ; Whisper-large-v3 seulement en LoRA 8 bits.
- Pas de multiples graines aléatoires : la variance d'entraînement n'est pas mesurée (seule la variance
  d'échantillonnage du test l'est, par bootstrap).

**Améliorations envisagées (non réalisées)**
- Données supplémentaires : corpus tunisiens ouverts (LinTO-TN, TunSwitch) et pseudo-étiquetage
  (self-training) d'heures de TEDx tunisien non annotées.
- LM neuronal (ou LLM) en rescoring des n-best au lieu d'un 4-gram.
- Évaluation hors domaine sur un second corpus tunisien.
- Plusieurs graines et moyenne des checkpoints.

## 13. Conclusion
Le code ci-dessous génère le résumé chiffré à partir des résultats effectivement obtenus.

In [ ]:
if len(asr_test) and BEST in set(asr_test["système"]):
    best_row = asr_test[asr_test["système"] == BEST].iloc[0]   # système retenu sur le dev, score rapporté sur le test
    zs = asr_test[asr_test["système"].str.contains("zeroshot")]
    print(f"Système retenu (choisi sur le dev), score de test : {best_row['système']} — WER {best_row['WER']:.2f} % "
          f"(IC95 {best_row['IC95 WER']}), CER {best_row['CER']:.2f} %, WER normalisé {best_row['WER normalisé']:.2f} %.")
    if len(zs):
        print(f"Gain par rapport au meilleur zero-shot : {zs['WER'].min() - best_row['WER']:.1f} points de WER.")
elif len(asr_test):
    print("Aucun système n'a été choisi sur le dev (pas de prédiction dev) : pas de conclusion chiffrée.")
print("Référence papier : w2v-BERT-2.0 21,37 % WER / 8,34 % CER.")

## Références
- F. Bougares, S. Mdhaffar, H. Elleuch, Y. Estève. *TEDxTN: A Three-way Speech Translation Corpus for
  Code-Switched Tunisian Arabic - English*. ArabicNLP 2025, p. 278-287.
- A. Radford et al. *Robust Speech Recognition via Large-Scale Weak Supervision* (Whisper), 2022.
- Seamless Communication et al. *Seamless: Multilingual Expressive and Streaming Speech Translation* (w2v-BERT 2.0), 2023.
- V. Pratap et al. *Scaling Speech Technology to 1,000+ Languages* (MMS), 2023.
- E. Hu et al. *LoRA: Low-Rank Adaptation of Large Language Models*, 2021.
- D. Park et al. *SpecAugment*, Interspeech 2019. T. Ko et al. *Audio augmentation for speech recognition*, Interspeech 2015.
- K. Heafield. *KenLM: Faster and Smaller Language Model Queries*, WMT 2011.
- A. Das, B. Gambäck. *Identifying Languages at the Word Level in Code-Mixed Indian Social Media Text* (CMI), 2014.
- P. Koehn. *Statistical Significance Tests for Machine Translation Evaluation*, EMNLP 2004.
- Bibliothèques : Hugging Face Transformers, PEFT, pyctcdecode, jiwer, sacreBLEU.